In [9]:
%load_ext autoreload
%autoreload 2
import sys; sys.path.append("..")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [10]:
import sys; sys.path.append("..")
import pandas as pd
from sklearn.model_selection import train_test_split

import src.config as cfg
from src.load_data import load_raw_data
from src.preprocess import clean_data, split_features_target
from src.train import (
    build_logistic_pipeline, build_random_forest_pipeline, train_model,
)
from src.evaluate import evaluate_classifier, print_classification_report

clean = clean_data(load_raw_data())
X, y = split_features_target(clean)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=cfg.TEST_SIZE, stratify=y, random_state=cfg.RANDOM_STATE,
)
print(f"Train: {X_train.shape}  |  Test: {X_test.shape}")
print(f"Train churn: {y_train.mean():.4f}  |  Test churn: {y_test.mean():.4f}")

Train: (5634, 22)  |  Test: (1409, 22)
Train churn: 0.2654  |  Test churn: 0.2654


In [11]:
lr, t_lr = train_model(build_logistic_pipeline(), X_train, y_train)
print(f"LR trained in {t_lr:.2f}s\n")

print("=== Logistic Regression — TEST SET ===")
print_classification_report(lr, X_test, y_test)

lr_test = evaluate_classifier(lr, X_test, y_test)
print("Metrics:", lr_test)

LR trained in 0.03s

=== Logistic Regression — TEST SET ===
              precision    recall  f1-score   support

    No churn     0.9084    0.7188    0.8026      1035
       Churn     0.5068    0.7995    0.6203       374

    accuracy                         0.7402      1409
   macro avg     0.7076    0.7592    0.7115      1409
weighted avg     0.8018    0.7402    0.7542      1409

Confusion matrix (rows=true, cols=pred):
        No    Yes
No     744    291
Yes     75    299
Metrics: {'threshold': 0.5, 'accuracy': 0.7402, 'precision': 0.5068, 'recall': 0.7995, 'f1': 0.6203, 'roc_auc': 0.8429, 'pr_auc': 0.6431}


In [12]:
rf, t_rf = train_model(build_random_forest_pipeline(), X_train, y_train)
print(f"RF trained in {t_rf:.2f}s\n")

print("=== Random Forest — TEST SET ===")
print_classification_report(rf, X_test, y_test)

rf_test = evaluate_classifier(rf, X_test, y_test)
print("Metrics:", rf_test)

RF trained in 1.26s

=== Random Forest — TEST SET ===
              precision    recall  f1-score   support

    No churn     0.8557    0.8135    0.8341      1035
       Churn     0.5459    0.6203    0.5807       374

    accuracy                         0.7622      1409
   macro avg     0.7008    0.7169    0.7074      1409
weighted avg     0.7735    0.7622    0.7668      1409

Confusion matrix (rows=true, cols=pred):
        No    Yes
No     842    193
Yes    142    232
Metrics: {'threshold': 0.5, 'accuracy': 0.7594, 'precision': 0.5404, 'recall': 0.6257, 'f1': 0.5799, 'roc_auc': 0.8227, 'pr_auc': 0.6149}


In [13]:
comparison = pd.DataFrame({
    "LogisticRegression": lr_test,
    "RandomForest": rf_test,
}).T
print(comparison)

                    threshold  accuracy  precision  recall      f1  roc_auc  \
LogisticRegression        0.5    0.7402     0.5068  0.7995  0.6203   0.8429   
RandomForest              0.5    0.7594     0.5404  0.6257  0.5799   0.8227   

                    pr_auc  
LogisticRegression  0.6431  
RandomForest        0.6149  


In [14]:
# What a dumb model looks like: predict all 'No'
naive_pred = pd.Series(0, index=y_test.index)
from sklearn.metrics import accuracy_score, recall_score, f1_score
print(f"Always-'No' baseline:  accuracy={accuracy_score(y_test, naive_pred):.4f}"
      f"  recall={recall_score(y_test, naive_pred):.4f}"
      f"  f1={f1_score(y_test, naive_pred):.4f}")
print("↑ This is what 73% accuracy looks like when it's useless.")

Always-'No' baseline:  accuracy=0.7346  recall=0.0000  f1=0.0000
↑ This is what 73% accuracy looks like when it's useless.


In [15]:
from src.evaluate import (
    plot_roc_pr_curves, plot_threshold_analysis, find_best_f1_threshold,
)

# Curves side by side
plot_roc_pr_curves(
    {"LogisticRegression": lr, "RandomForest": rf},
    X_test, y_test,
)

# Threshold analysis per model
for name, model in [("LogisticRegression", lr), ("RandomForest", rf)]:
    print(f"\n=== {name} ===")
    plot_threshold_analysis(model, X_test, y_test, name=name)

ImportError: cannot import name 'plot_roc_pr_curves' from 'src.evaluate' (/home/kakarotbot/telco-churn-prediction/notebooks/../src/evaluate.py)